# R19-H226 - Feature-ownership attachment census

**Hypothesis** - a feature-ownership census (per product: features its home documents state it has, vs features attached to its node in the reference graph) shows (a) >= 20% of document-stated ownerships missing from the owning node, and (b) the misses concentrate in multi-product documents.

**Method** - build a document-stated ownership reference BLIND from source documents (frozen before any graph query), then diff against neo4j2 (READ-ONLY, fingerprint-asserted). Each (product, stated feature) is scored attached / missing / attached-to-wrong-product; misses are partitioned by mechanism (never extracted / extracted-attached-elsewhere / extracted-unattached). CPU-only, no LLM.

**Bar** - clause (a) >= 20% missing; clause (b) misses concentrate in multi-product docs; refuted if attachment > 90% complete.

## Imports

In [1]:
import os, re, json, datetime, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict
os.environ["CUDA_VISIBLE_DEVICES"] = ""            # CPU-only
import fitz
from dotenv import dotenv_values
from neo4j import GraphDatabase
from rich import print as rprint

## Configuration - audited products, home-document anchors, feature lexicon

In [2]:
ROOT = Path("..")
PDFDIR = ROOT / "data/external/cpap-datasheets-and-manuals"
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NEO4J2 = "bolt://172.19.0.9:7687"                   # pinned baseline, READ-ONLY
EXPECT_RENDER_FP = "96ab16d299fbbc71"
LOG = ROOT / "logs/h226-h227-forensics.log"
def log(msg):
    with open(LOG, "a") as f:
        f.write(f"{datetime.datetime.now(datetime.timezone.utc).isoformat()} [H226] {msg}\n")

# 13 products: 8 single-product manuals + 5 multi-product catalogues (product_node, home_doc, class)
PRODUCTS = [
 ("AirSense 11 AutoSet","ResMed-Airsense-11-Manual.pdf","single"),
 ("AirStart 10 CPAP","airstart-10-cpap_fact-sheet_apac_eng.pdf","single"),
 ("DreamStation CPAP","DreamStation_CPAP_User_Manual.pdf","single"),
 ("DreamStation CPAP Pro","DreamStation_CPAP_Pro_DataSheet.pdf","single"),
 ("RESmart Auto CPAP System","BMC_RESmart_AutoCPAP_User_Manual.pdf","single"),
 ("SleepStyle 200 Series","SleepStyle_200_Operating_Manual.pdf","single"),
 ("iBreeze CPAP System","Resvent-iBreeze-Auto-CPAP-User-Manual.pdf","single"),
 ("RESmart CPAP","3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf","single"),
 ("prisma SOFT plus","PrismaSmart-and-Soft-Max-Brochure.pdf","multi"),
 ("AirSense 10 Elite","1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf","multi"),
 ("ResMed AirSense 10 AutoSet","Sleep And Respiratory Medical Devices Brochure.pdf","multi"),
 ("REMstar Auto","product_and_solutions_catalog.pdf","multi"),
 ("ELK-200C CPAP System","CPAP-Machines-Brochure.pdf","multi"),
]
# proximity anchors for multi-product docs (verified to occur in the home doc)
ANCHOR = {"prisma SOFT plus":["prisma soft"], "AirSense 10 Elite":["airsense 10 elite","10 elite"],
 "ResMed AirSense 10 AutoSet":["autoset"], "REMstar Auto":["remstar auto"], "ELK-200C CPAP System":["elk-200c","elk-200"]}

# canonical named features: (doc_regex, graph_name_substrings, feature-encoding prop_* substrings)
FEATS = {
 "ramp":("\\bramp\\b|auto ?ramp|smart ?ramp|opti-?start",["ramp","autoramp","smart ramp","opti-start"],["ramp"]),
 "pressure_relief":("\\bepr\\b|expiratory pressure relief|[abcp]-?flex\\b|soft ?pap|reslex|pressure relief|\\bflex\\b",["epr","flex","pressure relief","softpap","reslex","expiratory pressure relief"],["flex_pressure_relief"]),
 "ez_start":("ez-?start",["ez-start"],[]),
 "auto_onoff":("smart ?start|smart ?stop|auto-?start|auto-?stop|auto on/off|auto on|auto off",["smartstart","smartstop","auto on","auto off","auto-on","auto-off","auto start","auto stop","autostart"],[]),
 "humidification":("humidif|humidair|heated humidification|water chamber|prismaaqua",["humidif","humidair","heated respiratory humidification","heated humidification","water chamber","prismaaqua"],["humidif","humidification","has_humidifier"]),
 "heated_tube":("heated tube|heated breathing tube|climate ?line|thermosmart|heated hose|hybernite",["climatelineair","heated tube","heated breathing tube","thermosmart","hybernite","climateline"],["heated_tube"]),
 "bluetooth":("bluetooth",["bluetooth"],[]),
 "wifi":("\\bwi-?fi\\b|wireless network|wireless connectivity",["wifi","wi-fi","wireless connectivity"],["wireless"]),
 "cellular_modem":("cellular|gprs|\\b4g\\b|\\b3g\\b|built-in modem|cellular modem",["cellular","gprs","modem"],["modem_capable","wireless_modem_capable","broadband_modem","wired_modem"]),
 "sd_card":("sd card|sd-card|secure digital",["sd card","sd-card"],["sd_card"]),
 "oximetry":("oximet|spo2|\\bipom\\b|oximeter",["oximet","ipom","oximeter","spo2"],["oximetry"]),
 "auto_adjust":("auto-?adjust|\\bapap\\b|autoset|auto ?cpap|automatic positive|auto mode",["apap","autoset","auto mode","auto-adjust"],[]),
 "altitude_comp":("altitude compensat|automatic altitude|altitude adjust|altitude setting",["altitude compensation","altitude setting","altitude"],["altitude_compensation","altitude_setting"]),
 "mask_fit":("mask fit|check mask fit|fit check",["mask fit","check mask fit"],[]),
 "app_monitoring":("myair|dream ?mapper|prisma ?app|\\bicode\\b|remote monitoring",["myair","dreammapper","prismaapp","icode","remote monitoring"],[]),
}
FEATREL = ["HAS_FEATURE","HAS_COMFORT_FEATURE","HAS_CLINICAL_FEATURE","SUPPORTS_MODE","USES_ACCESSORY","PROVIDES","DISPLAYS","CONTROLS","USES_COMFORT_FEATURE","USES_FEATURE","ENABLES"]
def norm(t):
    t = unicodedata.normalize("NFKC", t or "")
    for a, b in {"\u2013":"-","\u2014":"-","\u2082":"2"}.items(): t = t.replace(a, b)
    return re.sub(r"\s+", " ", t).lower()
rprint(f"[cyan]config[/cyan] products={len(PRODUCTS)} (single={sum(1 for p in PRODUCTS if p[2]=='single')} multi={sum(1 for p in PRODUCTS if p[2]=='multi')}) features={len(FEATS)} stamp={STAMP}")

config products=13 (single=8 multi=5) features=15 stamp=20260707T204912Z

## Step 1 - BLIND document-stated ownership reference (frozen before any graph query)

For single-product manuals every stated feature is owned by the one device (whole-document scope). For multi-product catalogues ownership is proximity-scoped to +-750 chars around each home-anchor occurrence. Each ownership carries the document evidence string.

In [3]:
DOCTXT = {p.name: norm("\n".join(pg.get_text() for pg in fitz.open(p))) for p in PDFDIR.glob("*.pdf")}
def evidence(text, pat):
    m = re.search(pat, text)
    return None if not m else text[max(0, m.start()-55):min(len(text), m.end()+55)].strip()
def stated_features(prod, doc, cls):
    txt = DOCTXT[doc]
    if cls == "multi":
        wins = [txt[max(0,m.start()-750):m.end()+750] for al in ANCHOR[prod] for m in re.finditer(re.escape(al), txt)]
        scope = " ".join(wins)
    else:
        scope = txt
    return {fk: evidence(scope, pat) for fk, (pat, _, _) in FEATS.items() if evidence(scope, pat)}

STATED = {name: stated_features(name, doc, cls) for name, doc, cls in PRODUCTS}     # FROZEN reference
n_stated = sum(len(v) for v in STATED.values())
rprint(f"[green]blind reference frozen[/green] {n_stated} document-stated ownerships across {len(PRODUCTS)} products")
for name, doc, cls in PRODUCTS:
    rprint(f"  [{cls[0].upper()}] {name[:30]:30s} {sorted(STATED[name])}")
log(f"blind reference frozen: {n_stated} ownerships over {len(PRODUCTS)} products")

MuPDF error: format error: No default Layer config



blind reference frozen 92 document-stated ownerships across 13 products

[S] AirSense 11 AutoSet            ['auto_adjust', 'auto_onoff', 'bluetooth', 'cellular_modem', 'heated_tube', 
'humidification', 'mask_fit', 'oximetry', 'pressure_relief', 'ramp', 'sd_card', 'wifi']

[S] AirStart 10 CPAP               ['humidification']

[S] DreamStation CPAP              ['altitude_comp', 'app_monitoring', 'auto_adjust', 'bluetooth', 
'cellular_modem', 'ez_start', 'heated_tube', 'humidification', 'mask_fit', 'oximetry', 'pressure_relief', 'ramp', 
'sd_card', 'wifi']

...

[M] prisma SOFT plus               ['altitude_comp', 'app_monitoring', 'auto_adjust', 'bluetooth']

[M] AirSense 10 Elite              ['app_monitoring', 'auto_adjust', 'heated_tube', 'humidification', 
'pressure_relief', 'ramp', 'wifi']

[M] ResMed AirSense 10 AutoSet     ['auto_adjust', 'bluetooth', 'heated_tube', 'humidification', 
'pressure_relief', 'ramp', 'wifi']

[M] REMstar Auto                   ['heated_tube', 'humidification', 'oximetry', 'pressure_relief', 'ramp', 
'sd_card']

[M] ELK-200C CPAP System           ['humidification']

## Step 2 - graph pull (neo4j2 READ-ONLY): node attachments, siblings, home-doc feature entities

In [4]:
env = dotenv_values(ROOT / ".env"); AUTH = ("neo4j", env["NEO4J_PASSWORD"])
dr = GraphDatabase.driver(NEO4J2, auth=AUTH)
with dr.session() as s:
    DOCMAP = {r["id"]: r["name"] for r in s.run("MATCH (n:KGFDocument) RETURN n.id AS id, n.name AS name")}
    DOCID = {v: k for k, v in DOCMAP.items()}
    NODE, SIB, HOMEFEAT = {}, {}, {}
    for name, doc, cls in PRODUCTS:
        rec = s.run("MATCH (e:Entity {name:$n}) RETURN e.source_documents AS docs, [k IN keys(e) WHERE k STARTS WITH 'prop_'] AS props LIMIT 1", n=name).single()
        rels = s.run("MATCH (e:Entity {name:$n})-[r]-(f:Entity) WHERE type(r) IN $rt RETURN collect(DISTINCT toLower(f.name)) AS fn", n=name, rt=FEATREL).single()
        NODE[name] = dict(props=[k.lower() for k in (rec["props"] if rec else [])], rels=(rels["fn"] if rels else []), docs=(rec["docs"] if rec else []))
        sib = s.run("""MATCH (e:Entity) WHERE any(l IN labels(e) WHERE l IN ['CPAPDevice','ProductModel'])
            AND any(x IN e.source_documents WHERE x IN $d) AND e.name<>$n
            RETURN e.name AS n, [(e)-[r]-(f:Entity) WHERE type(r) IN $rt | toLower(f.name)] AS fn,
            [k IN keys(e) WHERE k STARTS WITH 'prop_'] AS props""", d=NODE[name]["docs"], n=name, rt=FEATREL).data()
        SIB[name] = [(x["n"], set(x["fn"]), [k.lower() for k in x["props"]]) for x in sib]
        did = DOCID.get(doc)
        hf = s.run("""MATCH (f:Entity) WHERE any(l IN labels(f) WHERE l IN ['Feature','ComfortFeature','ClinicalFeature','OperatingMode','ConnectivityDevice','Software'])
            AND $did IN f.source_documents
            RETURN toLower(f.name) AS n, [(f)-[r]-(p:Entity) WHERE any(l IN labels(p) WHERE l IN ['CPAPDevice','ProductModel']) AND type(r) IN $rt | p.name] AS owners""", did=did, rt=FEATREL).data()
        HOMEFEAT[name] = [(x["n"], x["owners"]) for x in hf]
    allprod = s.run("""MATCH (e:Entity)-[r]-(f:Entity) WHERE any(l IN labels(e) WHERE l IN ['CPAPDevice','ProductModel'])
        AND type(r) IN $rt RETURN e.name AS p, collect(DISTINCT toLower(f.name)) AS fn""", rt=FEATREL).data()
    PROD_FEAT = {r["p"]: set(r["fn"]) for r in allprod}
dr.close()
rprint(f"[green]pulled neo4j2[/green] {len(NODE)} product nodes, sibling sets, home-doc feature entities")

pulled neo4j2 13 product nodes, sibling sets, home-doc feature entities

## Step 3 - diff: attached / missing / attached-to-wrong-product, with mechanism partition

In [5]:
def match_names(names, coll): return any(any(g in rn for g in names) for rn in coll)
def match_props(propk, props): return bool(propk) and any(any(pk in pr for pk in propk) for pr in props)
def node_has(name, fk):
    _, gn, pk = FEATS[fk]; return match_names(gn, NODE[name]["rels"]) or match_props(pk, NODE[name]["props"])
def on_sibling(name, fk):
    _, gn, pk = FEATS[fk]
    for sn, fns, props in SIB[name]:
        if match_names(gn, fns) or match_props(pk, props): return sn
    return None

results, part = [], {"never_extracted":0, "attached_elsewhere_sibling":0, "attached_elsewhere_other":0, "extracted_unattached":0}
tot = miss = st = sm = mt = mm = 0
for name, doc, cls in PRODUCTS:
    sibnames = {x[0] for x in SIB[name]}; hf = HOMEFEAT[name]; rows = []
    for fk, ev in STATED[name].items():
        if node_has(name, fk):
            rows.append(dict(feature=fk, status="attached", mechanism=None, other_owner=None, evidence=ev[:110])); continue
        # home-doc-scoped mechanism partition
        _, gn, _ = FEATS[fk]
        cand = [(en, ow) for en, ow in hf if any(g in en for g in gn)]
        sib = on_sibling(name, fk)
        allow = [o for en, ow in cand for o in ow]
        if any(len(ow) == 0 for en, ow in cand) and not allow:
            mech, other = "extracted_unattached", None
        elif sib or any(o in sibnames for o in allow):
            mech, other = "attached_elsewhere_sibling", (sib or next(o for o in allow if o in sibnames))
        elif allow:
            mech, other = "attached_elsewhere_other", allow[0]
        elif cand:
            mech, other = "extracted_unattached", None
        else:
            mech, other = "never_extracted", None
        part[mech] += 1
        status = "attached_to_wrong_product" if "elsewhere" in mech else "missing"
        rows.append(dict(feature=fk, status=status, mechanism=mech, other_owner=other, evidence=ev[:110]))
    n = len(rows); m = sum(1 for r in rows if r["status"] != "attached"); tot += n; miss += m
    if cls == "single": st += n; sm += m
    else: mt += n; mm += m
    results.append(dict(product=name, home_doc=doc, doc_class=cls, n_stated=n, n_missing=m, rows=rows))
    rprint(f"  [{cls[0].upper()}] {name[:28]:28s} stated={n:2d} missing={m:2d} ({(m/n*100 if n else 0):3.0f}%)")
log(f"diff: {miss}/{tot} missing; single={sm}/{st} multi={mm}/{mt}; mechanism={part}")

[S] AirSense 11 AutoSet          stated=12 missing=11 ( 92%)

[S] AirStart 10 CPAP             stated= 1 missing= 0 (  0%)

[S] DreamStation CPAP            stated=14 missing= 7 ( 50%)

[S] DreamStation CPAP Pro        stated=11 missing= 4 ( 36%)

[S] RESmart Auto CPAP System     stated= 8 missing= 5 ( 62%)

...

[M] prisma SOFT plus             stated= 4 missing= 3 ( 75%)

[M] AirSense 10 Elite            stated= 7 missing= 1 ( 14%)

[M] ResMed AirSense 10 AutoSet   stated= 7 missing= 2 ( 29%)

[M] REMstar Auto                 stated= 6 missing= 0 (  0%)

[M] ELK-200C CPAP System         stated= 1 missing= 0 (  0%)

## Step 4 - clause evaluation and verdict

In [6]:
miss_rate = miss / tot
single_rate = sm / st; multi_rate = mm / mt
clause_a = miss_rate >= 0.20
clause_b = multi_rate > single_rate                       # "misses concentrate in multi-product docs"
refuted_overall = miss_rate < 0.10                        # attachment > 90% complete
rprint(f"[bold]TOTAL[/bold] {miss}/{tot} document-stated ownerships missing = [yellow]{miss_rate:.1%}[/]")
rprint(f"[bold]clause (a)[/bold] >=20% missing: {miss_rate:.1%} -> [{'PASS' if clause_a else 'FAIL'}]")
rprint(f"[bold]clause (b)[/bold] misses concentrate in MULTI-product docs: single={single_rate:.1%} vs multi={multi_rate:.1%} -> [{'PASS' if clause_b else 'FAIL (concentrate in SINGLE-product manuals)'}]")
rprint(f"[bold]refuted-if->90%-complete[/bold]: attachment={1-miss_rate:.1%} complete -> refuted branch fires: {refuted_overall}")
rprint(f"[bold]mechanism partition of {miss} misses[/bold]: {part}")
verdict = ("REFUTED" if refuted_overall else
           "CONFIRMED-PARTIAL (clause a CONFIRMED; clause b REFUTED)" if clause_a and not clause_b else
           "CONFIRMED" if clause_a and clause_b else "REFUTED")
rprint(f"[bold magenta]verdict recommendation[/bold magenta] {verdict}")
log(f"clause_a={clause_a} clause_b={clause_b} refuted={refuted_overall} verdict={verdict}")

TOTAL 38/92 document-stated ownerships missing = 41.3%

clause (a) >=20% missing: 41.3% -> [PASS]

clause (b) misses concentrate in MULTI-product docs: single=47.8% vs multi=24.0% -> [FAIL (concentrate in 
SINGLE-product manuals)]

refuted-if->90%-complete: attachment=58.7% complete -> refuted branch fires: False

mechanism partition of 38 misses: {'never_extracted': 12, 'attached_elsewhere_sibling': 17, 
'attached_elsewhere_other': 3, 'extracted_unattached': 6}

verdict recommendation CONFIRMED-PARTIAL (clause a CONFIRMED; clause b REFUTED)

## Step 5 - fingerprint assertion (pinned baseline) + machine-readable report

In [7]:
dr = GraphDatabase.driver(NEO4J2, auth=AUTH)
with dr.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id,e.name AS name,e.description AS description,properties(e) AS props,labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id RETURN DISTINCT a.id AS a,b.id AS b,type(r) AS rel").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run("MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
dr.close()
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges: rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))
def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a,'') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid):
    rels = "; ".join(f"{t} -> {names.get(b,'')}" for t, b in rels_by.get(nid, [])[:15])
    return base_render(nid) + "\nRelations: " + rels
CANON_SPEC = dict(source="pipeline._retrieve_local entity_blocks (production query path)",
    per_seed=["## name (types)","Also known as (SAME_AS*1..2, <=5)","description","Properties: json(prop_* keys, alias-merged)","Relations: type -> name (<=15, non-SIMILAR_TO)"],
    proposition_channel="per-seed attached propositions (Proposition-[:ABOUT]->seed)",
    eval_k=64, retrieve_top_k=128, rel_limit=15,
    seed_order="score desc, id asc (deterministic tie-break)",
    scorer="deterministic: exact_present(numeric/code) OR word_overlap>=0.6(prose); CPU-only, no NLI")
ALL = sorted(node)
render_fp = hashlib.sha256((json.dumps({k: CANON_SPEC[k] for k in sorted(CANON_SPEC)}, default=str) + "\x1e" +
    "\x1e".join(seed_render(c) for c in ALL)).encode()).hexdigest()[:16]
rb = "\x1e".join(seed_render(c) for c in ALL)
eb = ";".join(f"{c}:" + ",".join(f"{x:.4f}" for x in emb_head.get(c, [])) for c in ALL)
graph_fp = dict(node_count=len(node), edge_count=len(edges), embedding_count=len(emb_head),
    content_hash=hashlib.sha256(rb.encode()).hexdigest()[:16], embedding_digest=hashlib.sha256(eb.encode()).hexdigest()[:16])
assert render_fp == EXPECT_RENDER_FP, f"render fingerprint drift: {render_fp} != {EXPECT_RENDER_FP}"
rprint(f"[magenta]fingerprint asserted[/magenta] render={render_fp} (==expected) graph={graph_fp}")

report = dict(hypothesis="R19-H226", stamp=STAMP, driver=NEO4J2, mode="READ-ONLY",
    n_products=len(PRODUCTS), n_stated_ownerships=tot, n_missing=miss, miss_rate=miss_rate,
    attachment_complete=1-miss_rate,
    single=dict(tot=st, miss=sm, rate=single_rate), multi=dict(tot=mt, miss=mm, rate=multi_rate),
    clause_a=dict(threshold=0.20, value=miss_rate, result="PASS" if clause_a else "FAIL"),
    clause_b=dict(hypothesis="misses concentrate in multi-product docs", single_rate=single_rate, multi_rate=multi_rate, result="PASS" if clause_b else "FAIL"),
    refuted_if_over_90pct_complete=refuted_overall,
    mechanism_partition=part, verdict=verdict,
    frozen_reference={k: sorted(v) for k, v in STATED.items()},
    per_product=results, canon_features=list(FEATS),
    fingerprint=dict(render_fingerprint=render_fp, render_match=True, graph_fingerprint=graph_fp))
out = ROOT / "reports" / f"ownership-census-h226-{STAMP}.json"
out.write_text(json.dumps(report, indent=1))
rprint(f"[green]report written[/green] {out}")
log(f"report {out.name} verdict={verdict}")

fingerprint asserted render=96ab16d299fbbc71 (==expected) graph={'node_count': 2798, 'edge_count': 3905, 
'embedding_count': 2798, 'content_hash': '6fdc41bde495d1a3', 'embedding_digest': '2a3908456d2e2d8c'}

report written ../reports/ownership-census-h226-20260707T204912Z.json